# HOUSE PRICES

## Setup And Imports

In [ ]:
import numpy as np
import pandas as pd
import mlflow
import matplotlib.pyplot as plt
import dagshub
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_selection import SelectKBest, f_regression

import xgboost as xgb
import lightgbm as lgb
from scipy.stats import skew

dagshub.init(repo_owner='LukaBatilashvili07', repo_name='house-prices', mlflow=True)
mlflow.set_experiment('house-prices-experiment')

RANDOM_STATE = 42

## Data Loading

In [ ]:
train_data = pd.read_csv('data/train.csv')
test_data = pd.read_csv('data/test.csv')

print('Train shape:', train_data.shape)
print('Test shape:', test_data.shape)
train_data.head()

In [ ]:
missing_values = train_data.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
print("Missing values in training data:")
print(missing_values)

## Data Cleaning

In [ ]:
test_ids = test_data['Id']
all_data = pd.concat([train_data.drop('SalePrice', axis=1), test_data], axis=0).reset_index(drop=True)

none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 
            'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
            'MasVnrType']
for col in none_cols:
    all_data[col] = all_data[col].fillna('None')

zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
                'BsmtFullBath', 'BsmtHalfBath']
for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

mode_cols = ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType',
            'Functional', 'Utilities']
for col in mode_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

print("Missing values after imputation:")
print(all_data.isnull().sum()[all_data.isnull().sum() > 0])

In [ ]:
train_cleaned = train_data.copy()
outliers = train_cleaned[(train_cleaned['GrLivArea'] > 4000) & (train_cleaned['SalePrice'] < 300000)].index
train_cleaned = train_cleaned.drop(outliers)

y = np.log1p(train_cleaned['SalePrice'])
print('Target shape', y.shape)

## Feature Engineering

In [ ]:
def feature_engineering(df):
    df = df.copy()
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBath'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
    df['TotalPorchSF'] = df['OpenPorchSF'] + df['EnclosedPorch'] + df['3SsnPorch'] + df['ScreenPorch']
    df['Age'] = df['YrSold'] - df['YearBuilt']
    df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
    df['HasPool'] = (df['PoolArea'] > 0).astype(int)
    df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
    df['HasBsmt'] = (df['TotalBsmtSF'] > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)
    df['IsRemodeled'] = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)

    qual_mapping = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
    qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 
                'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']
    for col in qual_cols:
        if col in df.columns:
            df[col] = df[col].map(qual_mapping).fillna(0)

    categorical_cols = df.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        df[col] = df[col].astype('category').cat.codes
    return df

all_data_fe = feature_engineering(all_data)
print('Data shape after feature engineering:', all_data_fe.shape)

In [ ]:
len_train = len(train_cleaned)
X_full = all_data_fe[:len_train].copy()
X_test = all_data_fe[len_train:].copy()

X_train, X_val, y_train, y_val = train_test_split(X_full, y, test_size=0.2, random_state=RANDOM_STATE)

numeric_features = X_train.select_dtypes(include=[np.number]).columns
skewed = X_train[numeric_features].apply(lambda x: skew(x.dropna()))
skewed_features = skewed[skewed > 0.75].index
X_train[skewed_features] = np.log1p(X_train[skewed_features])
X_val[skewed_features] = np.log1p(X_val[skewed_features])
X_test[skewed_features] = np.log1p(X_test[skewed_features])

## Feature Selection

In [ ]:
rf_selector = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf_selector.fit(X_train, y_train)
rf_importances = pd.Series(rf_selector.feature_importances_, index=X_train.columns)
rf_selected_features = rf_importances[rf_importances > 0.01].index
print(len(rf_selected_features), "features selected")

X_train = X_train[rf_selected_features]
X_val = X_val[rf_selected_features]
X_test_selected = X_test[rf_selected_features]

## Model Training And MLFlow Logging

In [ ]:
def evaluate_model(model, X_train, y_train, X_val, y_val):
    train_preds = model.predict(X_train)
    val_preds = model.predict(X_val)
    train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
    val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    train_r2 = r2_score(y_train, train_preds)
    val_r2 = r2_score(y_val, val_preds)
    return train_rmse, val_rmse, train_r2, val_r2

results = {}

In [ ]:
with mlflow.start_run(run_name='Linear Regression'):
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    train_rmse, val_rmse, train_r2, val_r2 = evaluate_model(lr, X_train, y_train, X_val, y_val)
    mlflow.log_metric('train_rmse', train_rmse)
    mlflow.log_metric('val_rmse', val_rmse)
    mlflow.log_metric('train_r2', train_r2)
    mlflow.log_metric('val_r2', val_r2)
    mlflow.sklearn.log_model(lr, 'linear_regression_model')
    results['Linear Regression'] = {'train_rmse': train_rmse, 'val_rmse': val_rmse, 'train_r2': train_r2, 'val_r2': val_r2}
    print(f'Linear Regression - Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}, Train R2: {train_r2:.4f}, Val R2: {val_r2:.4f}')

In [ ]:
for alpha in [0.1, 1.0, 10.0]:
    with mlflow.start_run(run_name=f'Lasso alpha={alpha}'):
        lasso = Lasso(alpha=alpha, random_state=RANDOM_STATE)
        lasso.fit(X_train, y_train)
        train_rmse, val_rmse, train_r2, val_r2 = evaluate_model(lasso, X_train, y_train, X_val, y_val)
        mlflow.log_metric('train_rmse', train_rmse)
        mlflow.log_metric('val_rmse', val_rmse)
        mlflow.log_metric('train_r2', train_r2)
        mlflow.log_metric('val_r2', val_r2)
        mlflow.sklearn.log_model(lasso, 'lasso_model')
        results[f'Lasso alpha={alpha}'] = {'train_rmse': train_rmse, 'val_rmse': val_rmse, 'train_r2': train_r2, 'val_r2': val_r2}
        print(f'Lasso alpha={alpha} - Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}, Train R2: {train_r2:.4f}, Val R2: {val_r2:.4f}')

In [ ]:
for alpha in [0.1, 1.0, 10.0]:
    with mlflow.start_run(run_name=f'Ridge alpha={alpha}'):
        ridge = Ridge(alpha=alpha)
        ridge.fit(X_train, y_train)
        train_rmse, val_rmse, train_r2, val_r2 = evaluate_model(ridge, X_train, y_train, X_val, y_val)
        mlflow.log_metric('train_rmse', train_rmse)
        mlflow.log_metric('val_rmse', val_rmse)
        mlflow.log_metric('train_r2', train_r2)
        mlflow.log_metric('val_r2', val_r2)
        mlflow.sklearn.log_model(ridge, 'ridge_model')
        results[f'Ridge alpha={alpha}'] = {'train_rmse': train_rmse, 'val_rmse': val_rmse, 'train_r2': train_r2, 'val_r2': val_r2}
        print(f'Ridge alpha={alpha} - Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}, Train R2: {train_r2:.4f}, Val R2: {val_r2:.4f}')

In [ ]:
with mlflow.start_run(run_name='Random Forest'):
    rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
    rf.fit(X_train, y_train)
    train_rmse, val_rmse, train_r2, val_r2 = evaluate_model(rf, X_train, y_train, X_val, y_val)
    mlflow.log_metric('train_rmse', train_rmse)
    mlflow.log_metric('val_rmse', val_rmse)
    mlflow.log_metric('train_r2', train_r2)
    mlflow.log_metric('val_r2', val_r2)
    mlflow.sklearn.log_model(rf, 'random_forest_model')
    results['Random Forest'] = {'train_rmse': train_rmse, 'val_rmse': val_rmse, 'train_r2': train_r2, 'val_r2': val_r2}
    print(f'Random Forest - Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}, Train R2: {train_r2:.4f}, Val R2: {val_r2:.4f}')

In [ ]:
with mlflow.start_run(run_name='XGBoost'):
    xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=RANDOM_STATE)
    xgb_model.fit(X_train, y_train)
    train_rmse, val_rmse, train_r2, val_r2 = evaluate_model(xgb_model, X_train, y_train, X_val, y_val)
    mlflow.log_metric('train_rmse', train_rmse)
    mlflow.log_metric('val_rmse', val_rmse)
    mlflow.log_metric('train_r2', train_r2)
    mlflow.log_metric('val_r2', val_r2)
    mlflow.xgboost.log_model(xgb_model, 'xgboost_model')
    results['XGBoost'] = {'train_rmse': train_rmse, 'val_rmse': val_rmse, 'train_r2': train_r2, 'val_r2': val_r2}
    print(f'XGBoost - Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}, Train R2: {train_r2:.4f}, Val R2: {val_r2:.4f}')

In [ ]:
with mlflow.start_run(run_name='LightGBM'):
    lgb_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.1, max_depth=4, random_state=RANDOM_STATE)
    lgb_model.fit(X_train, y_train)
    train_rmse, val_rmse, train_r2, val_r2 = evaluate_model(lgb_model, X_train, y_train, X_val, y_val)
    mlflow.log_metric('train_rmse', train_rmse)
    mlflow.log_metric('val_rmse', val_rmse)
    mlflow.log_metric('train_r2', train_r2)
    mlflow.log_metric('val_r2', val_r2)
    mlflow.lightgbm.log_model(lgb_model, 'lightgbm_model')
    results['LightGBM'] = {'train_rmse': train_rmse, 'val_rmse': val_rmse, 'train_r2': train_r2, 'val_r2': val_r2}
    print(f'LightGBM - Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}, Train R2: {train_r2:.4f}, Val R2: {val_r2:.4f}')

In [ ]:
parameter_grid = {
    'n_estimators': [300, 500],
    'learning_rate': [0.03, 0.05],
    'max_depth': [3, 4],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_base = xgb.XGBRegressor(random_state=RANDOM_STATE)
xgb_random = RandomizedSearchCV(xgb_base, parameter_grid, n_iter=10, scoring='neg_root_mean_squared_error', random_state=RANDOM_STATE, cv=3)

xgb_random.fit(X_train, y_train)

best_params = xgb_random.best_params_
best_model = xgb_random.best_estimator_
train_rmse, val_rmse, train_r2, val_r2 = evaluate_model(best_model, X_train, y_train, X_val, y_val)

with mlflow.start_run(run_name='XGBoost Hyperparameter Tuning'):
    mlflow.log_params(best_params)
    mlflow.log_metric('train_rmse', train_rmse)
    mlflow.log_metric('val_rmse', val_rmse)
    mlflow.log_metric('train_r2', train_r2)
    mlflow.log_metric('val_r2', val_r2)
    mlflow.xgboost.log_model(best_model, 'model', registered_model_name='best_house_price_model')
    results['XGBoost Tuned'] = {'train_rmse': train_rmse, 'val_rmse': val_rmse, 'train_r2': train_r2, 'val_r2': val_r2}
    print(f'XGBoost Tuned - Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}, Train R2: {train_r2:.4f}, Val R2: {val_r2:.4f}')